# GraphVQA with Question-Conditioned Graph Construction

This notebook documents the full pipeline: dataset loading, preprocessing, model architecture, graph construction methods, training, and validation.

The baseline model is GraphVQA with a GAT reasoning module. We propose three question-conditioned graph construction methods that modify the scene graph structure before GNN message passing:
1. **Edge Re-Weighting** — gated-scores on existing edges
2. **Edge Augmentation** — add top-k new edges conditioned on the question
3. **Neighborhood Pruning** — keep only top-k neighbours per node

---

## 1. Imports and Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Data, Batch

from gqa_dataset_entry import GQATorchDataset, GQATorchDataset_collate_fn, GQA_gt_sg_feature_lookup
from pipeline_model_gat import PipelineModel
from graph_construction import (
    GraphMethod, QuestionConditionedGraphBuilder,
    apply_question_conditioned_graph_batched,
    EdgeReweighting, EdgeAugmentation, NeighborPruning
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## 2. Dataset

### 2.1 GQA Overview

We use the **GQA dataset**, which pairs images with compositional questions that require multi-step reasoning. Each sample contains:
- A natural-language **question** (e.g., *"What color is the chair to the left of the table?"*)
- A ground-truth **scene graph**: objects with attributes and pairwise spatial/semantic relations
- A structured **program**: up to 5 executable reasoning steps (e.g., `filter color: white`, `relate: left`)
- A **short answer** (one of 1,842 classes)
- A **full answer** (natural language)

We use the `train_unbiased` split for training and `val_unbiased` for evaluation.

### 2.2 `GQA_gt_sg_feature_lookup` — Scene Graph Loading

In [3]:
# GQA_gt_sg_feature_lookup loads and caches the ground-truth scene graphs.
# It builds a torchtext Field (SG_ENCODING_TEXT) over the scene-graph vocabulary
# (object names, attributes, relation labels) and initialises GloVe-300 vectors.
#
# Key class attribute:
#   SG_ENCODING_TEXT : torchtext Field
#     .vocab.stoi  : word → index
#     .vocab.itos  : index → word
#     .vocab.vectors : GloVe 300-d matrix  [V, 300]
#
# graph_method controls which graph topology is built at dataset time:
#   'static'          → symmetric scene-graph edges (baseline)
#   'fully_connected' → all N×N pairs
#   'reweight' / 'augment' / 'prune' → same as static; refinement happens in the model

GRAPH_METHOD = 'reweight'   # change to 'augment' | 'prune' | 'static' to switch method

sg_lookup = GQA_gt_sg_feature_lookup(split='val_unbiased', graph_method=GRAPH_METHOD)
print('SG vocab size:', len(GQA_gt_sg_feature_lookup.SG_ENCODING_TEXT.vocab))

SG vocab size: 2577


### 2.3 `GQATorchDataset` — Full Dataset

In [4]:
# GQATorchDataset wraps the GQA JSON files into a PyTorch Dataset.
# It builds a second torchtext Field (TEXT) for questions, programs, and full answers,
# with GloVe-300 initialisation.
#
# Each __getitem__ returns a 7-tuple:
#   questionID   : str
#   q_tokens     : LongTensor [Q_len]         question token indices
#   sg_datum     : torch_geometric.data.Data  scene graph
#     .x              [N, 12]   node token indices (1 name + up to 11 attrs)
#     .edge_index     [2, E]    COO directed edge list
#     .edge_attr      [E, 1]    relation token index per edge
#     .added_sym_edge [S]       indices of auto-added reverse edges
#     .graph_method   str       which construction method was tagged
#   prog_tokens  : LongTensor [P_len]         program token indices (all steps concatenated)
#   fa_tokens    : LongTensor [F_len]         full answer token indices
#   sa_label     : int                        short answer class index
#   types        : dict                       question type metadata

train_dataset = GQATorchDataset(
    split='train_unbiased',
    build_vocab_flag=False,
    load_vocab_flag=False,
    graph_method=GRAPH_METHOD,
)
val_dataset = GQATorchDataset(
    split='val_unbiased',
    build_vocab_flag=False,
    load_vocab_flag=True,
    graph_method=GRAPH_METHOD,
)

print(f'Train size : {len(train_dataset):,}')
print(f'Val size   : {len(val_dataset):,}')
print(f'QA vocab   : {len(GQATorchDataset.TEXT.vocab):,}')
print(f'Answer classes: {len(GQATorchDataset.ans2label):,}')

finished loading the data, totally 943000 instances
finished loading the data, totally 132062 instances
Train size : 943,000
Val size   : 132,062
QA vocab   : 3,657
Answer classes: 1,842


### 2.4 Inspecting One Sample

In [11]:
q_tok[0] in itos_qa

True

In [13]:
sample = train_dataset[42]
qID, q_tok, sg, prog, fa, sa_label, types = sample

# itos_sg = GQA_gt_sg_feature_lookup.SG_ENCODING_TEXT.vocab.itos
itos_sg = type(train_dataset.sg_feature_lookup).SG_ENCODING_TEXT.vocab.itos
itos_qa = GQATorchDataset.TEXT.vocab.itos

print('Question :', ' '.join(q_tok))
print('Answer   :', GQATorchDataset.label2ans[sa_label])
print()
print('Scene graph nodes:', sg.x.size(0))
print('Scene graph edges:', sg.edge_index.size(1))
print('graph_method tag :', sg.graph_method)
print()
print('First 3 nodes (name + attrs):')
for i in range(min(3, sg.x.size(0))):
    tokens = [itos_sg[t] for t in sg.x[i].tolist() if itos_sg[t] != '<pad>']
    print(f'  node {i}: {", ".join(tokens)}')

Question : Which kind of fast food is on the plate ?
Answer   : hot dog

Scene graph nodes: 34
Scene graph edges: 257
graph_method tag : reweight

First 3 nodes (name + attrs):
  node 0: shirt, striped, blue
  node 1: plate, white
  node 2: pants, black


### 2.5 DataLoader and Collation

`GQATorchDataset_collate_fn` merges a list of samples into a mini-batch. Scene graphs are batched using `torch_geometric.data.Batch.from_data_list`, which concatenates node/edge tensors and adds a `batch` vector mapping each node to its graph index. Question and program token sequences are padded to the longest sequence in the batch.

In [16]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    collate_fn=GQATorchDataset_collate_fn,
    pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    collate_fn=GQATorchDataset_collate_fn,
    pin_memory=True,
)

# Peek at one batched sample
qID_b, questions_b, sg_b, programs_b, fa_b, sa_label_b, types_b = next(iter(train_loader))
print('questions_b shape  :', questions_b.shape, '[Q_len, B]')
print('sg_b.x shape       :', sg_b.x.shape, '[N_total, 12]')
print('sg_b.edge_index    :', sg_b.edge_index.shape,'[2, E_total]')
print('sg_b.batch shape   :', sg_b.batch.shape, '[N_total]')
print('programs_b shape   :', programs_b.shape, '[P_len, B*5]')
print('sa_label_b shape   :', sa_label_b.shape, '[B]')

questions_b shape  : torch.Size([24, 128]) [Q_len, B]
sg_b.x shape       : torch.Size([2371, 12]) [N_total, 12]
sg_b.edge_index    : torch.Size([2, 11397]) [2, E_total]
sg_b.batch shape   : torch.Size([2371]) [N_total]
programs_b shape   : torch.Size([12, 640]) [P_len, B*5]
sa_label_b shape   : torch.Size([128]) [B]


## 3. Graph Construction

Scene graphs from GQA are human-annotated and not always complete or optimally structured for reasoning.
We propose three question-conditioned graph construction methods applied **after GloVe embedding, before the GNN**.
All methods share the same two-stage design:

- **Stage 1 (dataset time):** parse the raw scene graph JSON into a static PyG Data object.
- **Stage 2 (model forward time):** apply a learned, question-aware transformation to the embedded graph.

### 3.1 Stage 1 — Static Scene Graph Construction

The original scene graph is parsed once at dataset load time inside `convert_one_gqa_scene_graph`.

**Node features:** each object gets a token array of length 12 — index 0 is the object name, indices 1–11 are attributes. Tokens are looked up from the scene-graph vocabulary; unused slots are padded.

**Edge construction:**
- A **self-loop** (token `<self>`) is added for every node
- Each annotated relation `(i → j, label)` becomes a directed edge
- If the reverse edge `(j → i)` does not exist in the annotation, it is added automatically with the **same label but negated embedding** (sign flip applied in Stage 2) to signal directionality

The result is a symmetric graph where every annotated relation has both directions.

In [36]:
# Concrete example: look at the static graph for one sample
sg_datum = val_dataset[100][2]
itos_sg  = GQA_gt_sg_feature_lookup.SG_ENCODING_TEXT.vocab.itos

print(f'Nodes: {sg_datum.x.size(0)}')
print(f'Edges: {sg_datum.edge_index.size(1)}  (includes self-loops and symmetric fills)')
print(f'Auto-added symmetric edges: {sg_datum.added_sym_edge.size(0)}')
print()

for e in range(min(5, sg_datum.edge_index.size(1))):
    s = sg_datum.edge_index[0, e].item()
    d = sg_datum.edge_index[1, e].item()
    rel = itos_sg[sg_datum.edge_attr[e, 0].item()]
    print(f'  edge {e}: node {s} → node {d}  [{rel}]')

Nodes: 36
Edges: 128  (includes self-loops and symmetric fills)
Auto-added symmetric edges: 17

  edge 0: node 0 → node 0  [<self>]
  edge 1: node 0 → node 35  [above]
  edge 2: node 1 → node 1  [<self>]
  edge 3: node 1 → node 13  [next to]
  edge 4: node 1 → node 10  [to the right of]


### 3.2 Stage 2 — Question-Conditioned Graph Construction

Applied inside `GroundTruth_SceneGraph_Encoder.forward`, between the GloVe embedding step and the MetaLayer GNN.
The entry point is `apply_question_conditioned_graph_batched(gt_scene_graphs, h, e, q, builder)`.

**Shared MLP architecture:** all three methods use a 2-layer MLP (Linear → ReLU → Linear) to score pairs or edges.

#### Method 1 — Edge Re-Weighting

In [37]:
reweighter = EdgeReweighting(
    d_h=300,
    d_e=300,
    d_q=512,
    d_hidden=256,
)
print(reweighter)

h_demo = torch.randn(10, 300)
e_demo = torch.randn(15, 300)
q_demo = torch.randn(1, 512)
ei_demo = torch.randint(0, 10, (2, 15))
eb_demo = torch.zeros(15, dtype=torch.long)

weights = reweighter(h_demo, e_demo, q_demo, ei_demo, eb_demo)
print('Edge weights shape:', weights.shape, '  (one scalar per edge)')
print('Weight range: [{:.3f}, {:.3f}]'.format(weights.min().item(), weights.max().item()))

EdgeReweighting(
  (mlp_gate): MLP(
    (net): Sequential(
      (0): Linear(in_features=1112, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=1, bias=True)
    )
    (norm): Identity()
  )
  (mlp_score): MLP(
    (net): Sequential(
      (0): Linear(in_features=900, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=1, bias=True)
    )
    (norm): Identity()
  )
)
Edge weights shape: torch.Size([15])   (one scalar per edge)
Weight range: [0.173, 1.000]


#### Method 2 — Edge Augmentation

In [38]:
augmenter = EdgeAugmentation(
    d_h=300,
    d_q=512,
    d_hidden=256,
    top_k=5,
)
print(augmenter)

aug_ei, added = augmenter(h_demo, q_demo, ei_demo, node_batch=torch.zeros(10, dtype=torch.long))
print(f'Original edges: {ei_demo.size(1)},  After augmentation: {aug_ei.size(1)},  Added: {added}')

EdgeAugmentation(
  (mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=1112, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=1, bias=True)
    )
    (norm): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
)
Original edges: 15,  After augmentation: 20,  Added: 5


#### Method 3 — Neighbourhood Pruning

In [39]:
pruner = NeighborPruning(
    d_h=300,
    d_q=512,
    d_hidden=256,
    top_k=3,
    keep_self_loops=True,
)
print(pruner)

pruned_ei, kept_cols = pruner(h_demo, q_demo, ei_demo,
                               edge_batch=torch.zeros(15, dtype=torch.long))
print(f'Original edges: {ei_demo.size(1)},  After pruning: {pruned_ei.size(1)}')

NeighborPruning(
  (mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=1112, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=1, bias=True)
    )
    (norm): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
)
Original edges: 15,  After pruning: 15


### 3.3 Comparison of Graph Construction Methods

| | Topology | Input signals | Key inductive bias |
|---|---|---|---|
| **Static** | Unchanged | — | Scene graph is already sufficient |
| **Re-weight** | Unchanged | h_i, h_j, e_ij, q | Soft suppression of irrelevant edges |
| **Augment** | Grows by k edges | h_i, h_j, q | Annotations are incomplete |
| **Prune** | Shrinks to top-k/node | h_i, h_j, q | Dense graphs add noise |

## 4. Training

### 4.1 Loss Functions

Two losses are trained jointly:

- **`short_answer_loss`** — CrossEntropyLoss over 1,842 classes (standard softmax)
- **`program_loss`** — CrossEntropyLoss over the program token vocabulary, applied token-by-token over all 5 steps simultaneously

The full answer decoder (`TransformerFullAnswerDecoder`) is instantiated but its loss is disabled — it is dead code in the current configuration.

**Combined loss:**
```python
loss = short_answer_loss + program_loss
```

In [51]:
criterion = {
    'program':      nn.CrossEntropyLoss(ignore_index=GQATorchDataset.TEXT.vocab.stoi['<pad>']),
    'short_answer': nn.CrossEntropyLoss(),
}

model = PipelineModel(graph_method="reweight",
                      graph_top_k=5,
                      graph_hidden_dim=256).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

def text_generation_loss(loss_fn, output, target):
    V = len(GQATorchDataset.TEXT.vocab)
    return loss_fn(output.contiguous().view(-1, V),
                   target.contiguous().view(-1))

print('Criterion:', criterion)
print('Optimizer:', optimizer)

/home/gpuhead-2/genai_project/GraphVQA/pipeline_model_gat.py:540: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = torch.nn.TransformerEncoder(encoder_layers, nlayers, norm=torch.nn.LayerNorm(ninp) )


Criterion: {'program': CrossEntropyLoss(), 'short_answer': CrossEntropyLoss()}
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 1e-05
)


### 4.2 One Training Step (annotated)

In [46]:
def train_one_step(model, batch, criterion, optimizer):
    model.train()
    qID, questions, gt_scene_graphs, programs, full_answers, sa_label, types = batch

    questions = questions.to(DEVICE)
    gt_scene_graphs = gt_scene_graphs.to(DEVICE)
    programs = programs.to(DEVICE)
    sa_label = sa_label.to(DEVICE)

    programs_input = programs[:-1]
    programs_target = programs[1:]

    programs_output, sa_logits = model(
        questions = questions,
        gt_scene_graphs = gt_scene_graphs,
        programs_input = programs_input,
        full_answers_input = None,
        SAMPLE_FLAG = False,
    )

    program_loss = text_generation_loss(criterion['program'], programs_output, programs_target)
    short_answer_loss = criterion['short_answer'](sa_logits, sa_label)
    loss = short_answer_loss + program_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item(), short_answer_loss.item(), program_loss.item()

### 4.3 Training Loop

In [54]:
# Distributed training is handled by torch.nn.parallel.DistributedDataParallel
# in the main script (mainExplain_gat.py). The loop below shows the
# single-GPU version for clarity.

NUM_EPOCHS = 100

for epoch in range(NUM_EPOCHS):
    model.train()
    for i, batch in enumerate(train_loader):
        loss, sa_loss, prog_loss = train_one_step(model, batch, criterion, optimizer)

        if i % 100 == 0:
            print(f'Epoch {epoch}  step {i}  '
                  f'loss={loss:.4f}  sa={sa_loss:.4f}  prog={prog_loss:.4f}')
        
        if i != 0 and i % 1000 == 0:
            break
    scheduler.step()

    # torch.save({
    #     'epoch': epoch,
    #     'model': model.state_dict(),
    #     'optimizer': optimizer.state_dict(),
    # }, f'./outputdir/gat_{GRAPH_METHOD}/checkpoint{epoch:04d}.pth')

    print(f'Epoch {epoch} done. Checkpoint saved.')
    break

Epoch 0  step 0  loss=12.0475  sa=7.0608  prog=4.9867
Epoch 0  step 100  loss=4.7621  sa=3.7223  prog=1.0398
Epoch 0  step 200  loss=3.5322  sa=2.7764  prog=0.7558
Epoch 0  step 300  loss=3.7924  sa=3.2401  prog=0.5523
Epoch 0  step 400  loss=2.9542  sa=2.4766  prog=0.4776
Epoch 0  step 500  loss=2.8341  sa=2.4446  prog=0.3894
Epoch 0  step 600  loss=2.3470  sa=2.0119  prog=0.3351
Epoch 0  step 700  loss=2.7928  sa=2.5005  prog=0.2923
Epoch 0  step 800  loss=2.7592  sa=2.5183  prog=0.2409
Epoch 0  step 900  loss=2.4993  sa=2.2700  prog=0.2293
Epoch 0  step 1000  loss=2.0222  sa=1.8229  prog=0.1994
Epoch 0 done. Checkpoint saved.


## 5. Validation

Four metrics are reported:

| Metric | Definition |
|---|---|
| **Acc@Short** | Top-1 accuracy over 1,842 short answer classes |
| **Acc@Program** | Per-instruction token-level exact match (incl. empty steps) |
| **Acc@ProgramNonEmpty** | Same but excluding padding-only instruction slots |
| **Acc@ProgramGroup** | All 5 instruction steps correct simultaneously |

Acc@ProgramGroup is the strictest: one wrong token in any step → 0 for that sample.

### 5.1 Exact-Match Accuracy Helper

In [55]:
def program_string_exact_match_acc(predictions, target, padding_idx, group_accuracy_WAY_NUM):
    target_len = target.size(0)
    predictions = predictions[:target_len]

    char_match = (predictions == target).long()
    cond_match = torch.where(target == padding_idx,
                              torch.ones_like(target), char_match)

    match_reduced, _ = torch.min(cond_match, dim=0)

    per_step_acc = match_reduced.float().mean().item()

    B5 = match_reduced.size(0)
    B = B5 // group_accuracy_WAY_NUM
    grp = match_reduced.view(B, group_accuracy_WAY_NUM).min(dim=-1).values
    group_acc = grp.float().mean().item()

    empty_flag = (target[2] == padding_idx) & match_reduced.bool()
    non_empty_correct = match_reduced.sum().item() - empty_flag.sum().item()
    non_empty_total = B5 - (target[2] == padding_idx).sum().item()
    non_empty_acc = non_empty_correct / max(non_empty_total, 1)

    return per_step_acc, group_acc, non_empty_acc

### 5.2 Validation Loop

In [56]:
@torch.no_grad()
def validate(model, val_loader, criterion):
    model.eval()
    text_pad_idx = GQATorchDataset.TEXT.vocab.stoi['<pad>']

    total_sa_correct, total_prog_correct, total_group_correct = 0, 0, 0
    total_sa, total_prog = 0, 0

    for batch in val_loader:
        qID, questions, gt_sg, programs, fa, sa_label, types = batch

        questions = questions.to(DEVICE)
        gt_sg = gt_sg.to(DEVICE)
        programs = programs.to(DEVICE)
        sa_label = sa_label.to(DEVICE)

        programs_target = programs

        programs_output, sa_logits = model(
            questions = questions,
            gt_scene_graphs = gt_sg,
            programs_input = None,
            full_answers_input = None,
            SAMPLE_FLAG = True,
        )

        sa_pred = sa_logits.argmax(dim=-1)
        total_sa_correct += (sa_pred == sa_label).sum().item()
        total_sa += sa_label.size(0)

        prog_pred = programs_output.argmax(dim=-1)
        per_step, group, non_empty = program_string_exact_match_acc(
            prog_pred, programs_target, text_pad_idx,
            group_accuracy_WAY_NUM=GQATorchDataset.MAX_EXECUTION_STEP
        )
        B = sa_label.size(0)
        total_prog_correct += per_step * B * GQATorchDataset.MAX_EXECUTION_STEP
        total_group_correct += group * B
        total_prog += B * GQATorchDataset.MAX_EXECUTION_STEP

    results = {
        'Acc@Short' : total_sa_correct / total_sa,
        'Acc@Program' : total_prog_correct / total_prog,
        'Acc@ProgramGroup' : total_group_correct / total_sa,
    }
    for k, v in results.items():
        print(f'{k}: {100*v:.2f}%')
    return results

## 6. Loading a Checkpoint and Running Inference

Checkpoints are saved to `./outputdir/gat_{method}/checkpoint{epoch:04d}.pth`.
Each checkpoint stores the model state dict, optimizer state, and epoch number.

In [57]:
def load_checkpoint(checkpoint_path, graph_method='reweight', graph_top_k=5):
    model = PipelineModel(
        graph_method=graph_method,
        graph_top_k=graph_top_k,
        graph_hidden_dim=256,
    ).to(DEVICE)
    ckpt  = torch.load(checkpoint_path, map_location=DEVICE)
    state = ckpt.get('model', ckpt)
    model.load_state_dict(state, strict=False)
    model.eval()
    print(f'Loaded epoch {ckpt.get("epoch", "?")} from {checkpoint_path}')
    return model